<a href="https://colab.research.google.com/github/chaitanyafam-prog/dysarthria_screening/blob/main/notebookedac316cba_(2).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [6]:
from google.colab import drive

drive.mount('/content/drive')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [7]:
!pip install -q --upgrade kaggle

import os
from getpass import getpass
os.environ['KAGGLE_API_TOKEN'] = getpass("Paste new Kaggle token: ")

!kaggle datasets download -d didyoujustsaytanya7/dysarthria-model-training -p /content --unzip

Paste new Kaggle token: ··········
Dataset URL: https://www.kaggle.com/datasets/didyoujustsaytanya7/dysarthria-model-training
License(s): unknown
100% 12.6k/12.6k [00:00<00:00, 25.5MB/s]



In [8]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("aryashah2k/noise-reduced-uaspeech-dysarthria-dataset")

print("Path to dataset files:", path)

Path to dataset files: /root/.cache/kagglehub/datasets/aryashah2k/noise-reduced-uaspeech-dysarthria-dataset/versions/1


In [9]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("iamhungundji/dysarthria-detection")

print("Path to dataset files:", path)

Using Colab cache for faster access to the 'dysarthria-detection' dataset.
Path to dataset files: /kaggle/input/dysarthria-detection


In [10]:
!pip install praat-parselmouth xgboost -q

import pandas as pd
import numpy as np
import os
import re
import librosa
import parselmouth
from parselmouth.praat import call
import warnings
warnings.filterwarnings('ignore')

In [11]:
!find /kaggle/input -iname "*uaspeech*" -maxdepth 4

find: warning: you have specified the global option -maxdepth after the argument -iname, but global options are not positional, i.e., -maxdepth affects tests specified before it as well as those specified after it.  Please specify global options before other arguments.


In [12]:
!ls "/kaggle/input/datasets/aryashah2k/noise-reduced-uaspeech-dysarthria-dataset/"

ls: cannot access '/kaggle/input/datasets/aryashah2k/noise-reduced-uaspeech-dysarthria-dataset/': No such file or directory


In [13]:
ua_root = '/kaggle/input/noise-reduced-uaspeech-dysarthria-dataset/'

import os

test_path = os.path.join(ua_root, 'noisereduced-uaspeech', 'F02', 'F02_B1_C11_M5.wav')
print(os.path.exists(test_path))

control_check = os.path.join(ua_root, 'noisereduced-uaspeech-control')
print(os.path.exists(control_check))
if os.path.exists(control_check):
    print(os.listdir(control_check)[:5])

False
False


In [17]:
import glob
import os
import pandas as pd

# Pointing to the correct path downloaded by kagglehub in cell -WKkgKjpiNNv
ua_root = '/root/.cache/kagglehub/datasets/aryashah2k/noise-reduced-uaspeech-dysarthria-dataset/versions/1'
dys_base = os.path.join(ua_root, 'noisereduced-uaspeech')
control_base = os.path.join(ua_root, 'noisereduced-uaspeech-control')

dysarthric_speakers = ['F02','F03','F04','F05','M01','M04','M05','M07','M08','M09','M10','M11','M12','M14','M16']
control_speakers = ['CF02','CF03','CF04','CF05','CM01','CM04','CM05','CM06','CM08','CM09','CM10','CM12','CM13']

rows = []

if os.path.exists(dys_base) and os.path.exists(control_base):
    for speaker in dysarthric_speakers:
        files = glob.glob(os.path.join(dys_base, speaker, '*.wav'))
        for f in files:
            rows.append({'filename': f, 'is_dysarthria': 'dysarthria', 'speaker_id': speaker})

    for speaker in control_speakers:
        files = glob.glob(os.path.join(control_base, speaker, '*.wav'))
        for f in files:
            rows.append({'filename': f, 'is_dysarthria': 'non-dysarthria', 'speaker_id': speaker})
else:
    print(f"Directory warning: dys_base exists? {os.path.exists(dys_base)}, control_base exists? {os.path.exists(control_base)}")

ua_df = pd.DataFrame(rows)
print(ua_df.shape)
if not ua_df.empty:
    print(ua_df['speaker_id'].value_counts())
    print(ua_df['is_dysarthria'].value_counts())

(143290, 3)
speaker_id
F02     5355
F05     5355
M05     5355
M07     5355
M10     5355
M08     5355
M09     5355
CF05    5355
CF03    5355
CF04    5355
CF02    5355
M14     5355
CM09    5355
CM10    5355
CM06    5355
CM08    5355
CM12    5355
CM01    5355
CM05    5355
CM04    5355
CM13    5355
F04     5251
F03     5184
M11     4590
M12     4590
M16     4590
M04     3825
M01     2805
Name: count, dtype: int64
is_dysarthria
dysarthria        73675
non-dysarthria    69615
Name: count, dtype: int64


In [18]:
if ua_df.empty or 'speaker_id' not in ua_df.columns:
    print("Error: 'ua_df' is empty or does not contain 'speaker_id'. Please check your dataset path in cell h6-4OK-gJ31i.")
else:
    ua_df_sampled = ua_df.groupby('speaker_id', group_keys=False).apply(
        lambda x: x.sample(min(len(x), 100), random_state=42)
    )
    print(ua_df_sampled.shape)

(2800, 3)


FEATURE EXTRACTION -> COMMON FOR BOTH TORGO AND UASPEECH

In [19]:
def extract_rich_features(path, sr=16000):
    y, sr = librosa.load(path, sr=sr)

    # MFCC + deltas
    mfcc = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=13)
    delta = librosa.feature.delta(mfcc)
    delta2 = librosa.feature.delta(mfcc, order=2)
    mfcc_feats = np.concatenate([
        mfcc.mean(axis=1), mfcc.std(axis=1),
        delta.mean(axis=1), delta.std(axis=1),
        delta2.mean(axis=1), delta2.std(axis=1)
    ])  # 13*6 = 78 features

    # Basic spectral/energy
    zcr = librosa.feature.zero_crossing_rate(y).mean()
    rmse = librosa.feature.rms(y=y).mean()

    # Praat-based clinical features
    snd = parselmouth.Sound(path)
    pitch = snd.to_pitch()
    point_process = call(snd, "To PointProcess (periodic, cc)", 75, 500)

    try:
        jitter = call(point_process, "Get jitter (local)", 0, 0, 0.0001, 0.02, 1.3)
    except Exception:
        jitter = 0
    try:
        shimmer = call([snd, point_process], "Get shimmer (local)", 0, 0, 0.0001, 0.02, 1.3, 1.6)
    except Exception:
        shimmer = 0
    try:
        harmonicity = call(snd, "To Harmonicity (cc)", 0.01, 75, 0.1, 1.0)
        hnr = call(harmonicity, "Get mean", 0, 0)
    except Exception:
        hnr = 0

    # Formants (F1, F2 average)
    try:
        formant = call(snd, "To Formant (burg)", 0.0025, 5, 5500, 0.025, 50)
        f1 = call(formant, "Get mean", 1, 0, 0, "Hertz")
        f2 = call(formant, "Get mean", 2, 0, 0, "Hertz")
    except Exception:
        f1, f2 = 0, 0

    clinical_feats = np.array([jitter, shimmer, hnr, f1, f2, zcr, rmse])

    return np.concatenate([mfcc_feats, clinical_feats])  # 78 + 7 = 85 features

In [21]:
from tqdm import tqdm
import os
import numpy as np

ua_features, ua_labels, ua_speakers = [], [], []
fail_count = 0

for idx, row in tqdm(ua_df_sampled.iterrows(), total=len(ua_df_sampled)):
    try:
        feat = extract_rich_features(row['filename'])
        ua_features.append(feat)
        ua_labels.append(1 if row['is_dysarthria'] == 'dysarthria' else 0)
        ua_speakers.append(row['speaker_id'])
    except Exception as e:
        fail_count += 1
        if fail_count <= 5:  # only print the first few, not all 2800
            print(f"Failed on {row['filename']}: {e}")

print(f"Total failures: {fail_count} / {len(ua_df_sampled)}")

X_ua = np.array(ua_features)
y_ua = np.array(ua_labels)
groups_ua = np.array(ua_speakers)
print(X_ua.shape, y_ua.shape)

# Ensure the target directory exists
os.makedirs('/kaggle/working', exist_ok=True)

np.save('/kaggle/working/X_ua.npy', X_ua)
np.save('/kaggle/working/y_ua.npy', y_ua)
np.save('/kaggle/working/groups_ua.npy', groups_ua)

100%|██████████| 2800/2800 [13:23<00:00,  3.49it/s]

Total failures: 0 / 2800
(2800, 85) (2800,)


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [22]:
import os
import re
import pandas as pd

# Ensure output directory exists
os.makedirs('/kaggle/working', exist_ok=True)

df = pd.read_csv('/kaggle/input/dysarthria-detection/torgo_data/data.csv')
base = '/kaggle/input/dysarthria-detection/'
df['filename'] = df['filename'].apply(lambda x: os.path.join(base, x))

def get_speaker_id(path):
    fname = os.path.basename(path)
    match = re.match(r'([A-Z]+\d+)', fname)
    return match.group(1) if match else 'unknown'

df['speaker_id'] = df['filename'].apply(get_speaker_id)
print(df['speaker_id'].value_counts())

speaker_id
F03     292
FC03    231
FC02    230
MC01    171
MC03    153
F04     133
M05     127
M02     108
M03     100
MC04     96
M01      89
MC02     80
M04      76
F01      75
FC01     39
Name: count, dtype: int64


In [23]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os


# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All"
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [24]:
from tqdm import tqdm
import os
import numpy as np

features, labels, speakers = [], [], []
missing_files_count = 0

# Quick path check before loop
for idx, row in df.iterrows():
    if not os.path.exists(row['filename']):
        missing_files_count += 1

if missing_files_count > 0:
    print(f"Warning: {missing_files_count} out of {len(df)} files are missing at their specified paths!")
else:
    print("All TORGO audio file paths verified and exist on disk.")

for idx, row in tqdm(df.iterrows(), total=len(df)):
    try:
        if not os.path.exists(row['filename']):
            continue
        feat = extract_rich_features(row['filename'])
        features.append(feat)
        labels.append(1 if row['is_dysarthria'] == 'dysarthria' else 0)
        speakers.append(row['speaker_id'])
    except Exception as e:
        pass

X = np.array(features)
y = np.array(labels)
groups = np.array(speakers)

print(X.shape, y.shape)

# Ensure output directory exists
os.makedirs('/kaggle/working', exist_ok=True)
np.save('/kaggle/working/X_rich.npy', X)
np.save('/kaggle/working/y_rich.npy', y)
np.save('/kaggle/working/groups.npy', groups)

All TORGO audio file paths verified and exist on disk.


100%|██████████| 2000/2000 [09:20<00:00,  3.57it/s]

(1991, 85) (1991,)


In [25]:
import numpy as np
import pandas as pd
import os

required_files = [
    '/kaggle/working/X_rich.npy',
    '/kaggle/working/y_rich.npy',
    '/kaggle/working/groups.npy',
    '/kaggle/working/X_ua.npy',
    '/kaggle/working/y_ua.npy'
]

for filepath in required_files:
    if not os.path.exists(filepath):
        print(f"Warning: Required file {filepath} is missing!")
    else:
        print(f"Confirmed: {filepath} exists.")

X = np.load('/kaggle/working/X_rich.npy')
y = np.load('/kaggle/working/y_rich.npy')
groups = np.load('/kaggle/working/groups.npy')
X_ua = np.load('/kaggle/working/X_ua.npy')
y_ua = np.load('/kaggle/working/y_ua.npy')

print(X.shape, y.shape, groups.shape)
print(X_ua.shape, y_ua.shape)

Confirmed: /kaggle/working/X_rich.npy exists.
Confirmed: /kaggle/working/y_rich.npy exists.
Confirmed: /kaggle/working/groups.npy exists.
Confirmed: /kaggle/working/X_ua.npy exists.
Confirmed: /kaggle/working/y_ua.npy exists.
(1991, 85) (1991,) (1991,)
(2800, 85) (2800,)


In [26]:
import numpy as np
from sklearn.impute import SimpleImputer

print("NaNs found:", np.isnan(X).sum(), "across", np.isnan(X).any(axis=1).sum(), "rows")

imputer = SimpleImputer(strategy='median')
X = imputer.fit_transform(X)



NaNs found: 31 across 16 rows


looking for saved files in the working directory

In [ ]:
groups_ua = np.load('/kaggle/working/groups_ua.npy', allow_pickle=True)

imputer = SimpleImputer(strategy='median')
X_imputed = imputer.fit_transform(X)
X_ua_imputed = imputer.transform(X_ua)

X_combined = np.concatenate([X_imputed, X_ua_imputed])
y_combined = np.concatenate([y, y_ua])
groups_combined = np.concatenate([groups, groups_ua])

print(X_combined.shape, y_combined.shape, len(np.unique(groups_combined)), "unique speakers")

scaler_combined = StandardScaler()
X_combined_scaled = scaler_combined.fit_transform(X_combined)

logo = LeaveOneGroupOut()
all_preds_comb, all_true_comb = [], []

for train_idx, test_idx in logo.split(X_combined_scaled, y_combined, groups_combined):
    model = SVC(kernel='rbf', C=1.0, gamma='scale')
    model.fit(X_combined_scaled[train_idx], y_combined[train_idx])
    preds = model.predict(X_combined_scaled[test_idx])
    all_preds_comb.extend(preds)
    all_true_comb.extend(y_combined[test_idx])

print("=== Speaker-level LOGO-CV, TORGO+UASpeech combined ===")
print(classification_report(all_true_comb, all_preds_comb))

In [ ]:
import os
for f in ['X_features.npy','y_labels.npy','X_rich.npy','y_rich.npy','groups.npy','X_ua.npy','y_ua.npy']:
    print(f, os.path.exists(f'/kaggle/working/{f}'))

In [ ]:
from sklearn.model_selection import LeaveOneGroupOut, GridSearchCV
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report
from xgboost import XGBClassifier

X_clinical_only = X[:, 78:]  # jitter, shimmer, hnr, f1, f2, zcr, rmse
print(X_clinical_only.shape)

imputer_c = SimpleImputer(strategy='median')
X_clinical_only = imputer_c.fit_transform(X_clinical_only)

scaler_c = StandardScaler()
X_clinical_scaled = scaler_c.fit_transform(X_clinical_only)

all_preds_c, all_true_c = [], []
for train_idx, test_idx in logo.split(X_clinical_scaled, y, groups):
    model = SVC(kernel='rbf', C=1.0, gamma='scale')
    model.fit(X_clinical_scaled[train_idx], y[train_idx])
    preds = model.predict(X_clinical_scaled[test_idx])
    all_preds_c.extend(preds)
    all_true_c.extend(y[test_idx])

print("=== Speaker-level LOGO-CV (clinical features only) ===")
print(classification_report(all_true_c, all_preds_c))

In [ ]:
features = []
labels = []

for idx, row in df.iterrows():
    try:
        feat = extract_features(row['filename'])
        features.append(feat)
        labels.append(1 if row['is_dysarthria'] == 'dysarthria' else 0)
    except Exception as e:
        print(f"Failed on {row['filename']}: {e}")

X = np.array(features)
y = np.array(labels)

np.save('/kaggle/working/X_features.npy', X)
np.save('/kaggle/working/y_labels.npy', y)

really bad accuracy score with just torgo dataset, model simply remembered the people's voices, precision: 60% at best, so now im using UASpeech dataset and using both we'll see if there is any improvement

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, confusion_matrix
import joblib

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)

svm = SVC(kernel='rbf', probability=True)
svm.fit(X_train_s, y_train)
print("=== SVM ===")
print(classification_report(y_test, svm.predict(X_test_s)))

rf = RandomForestClassifier(n_estimators=200, random_state=42)
rf.fit(X_train, y_train)
print("=== Random Forest ===")
print(classification_report(y_test, rf.predict(X_test)))

In [ ]:
final_model = SVC(kernel='rbf', C=1.0, gamma='scale', probability=True)
final_model.fit(X_combined_scaled, y_combined)

import joblib
joblib.dump(final_model, '/kaggle/working/svm_final_combined.pkl')
joblib.dump(imputer, '/kaggle/working/imputer_final.pkl')
joblib.dump(scaler_combined, '/kaggle/working/scaler_final.pkl')

In [ ]:
import sklearn
print(sklearn.__version__)

In [ ]:
print(python.__version__)

In [ ]:
import sys
print(sys.version)


In [27]:
import sys; sys.path.insert(0, "/content/model_training")
exec(open("/content/model_training/step1_fix_baseline.py").read())

[    0s] workers: 2
samples 4791 | speakers by old IDs: 38 | by corpus-prefixed IDs: 43
colliding IDs: [np.str_('F03'), np.str_('F04'), np.str_('M01'), np.str_('M04'), np.str_('M05')]
[    0s] starting (a): about 1-3 minutes per stage
[   33s] (a) original protocol                       accuracy 0.745
[   65s] (b) + correct speaker IDs                   accuracy 0.745
[   65s] starting (c)
(c) + preprocessing inside each fold        accuracy 0.745  (95% CI 0.633-0.845)
    balanced acc 0.746 | AUC 0.824 | per-speaker mean 0.796 +/- 0.255, worst 0.068
    TORGO 0.553 | UASpeech 0.882
    hardest speakers: T_F03=0.07, T_F04=0.10, T_M03=0.29, T_MC04=0.30, U_M16=0.39, T_FC03=0.43, U_M14=0.47, U_CM05=0.54
[   99s] cross-corpus train torgo     -> test uaspeech : accuracy 0.536, AUC 0.845
[  100s] cross-corpus train uaspeech  -> test torgo    : accuracy 0.481, AUC 0.545


In [1]:
!pip install -q praat-parselmouth xgboost tqdm librosa

import sys
sys.path.insert(0, "/content/model_training")
exec(open("/content/model_training/step2_extract.py").read())

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.7/10.7 MB 64.8 MB/s eta 0:00:00


OSError: [Errno 30] Read-only file system: '/kaggle/input/datasets'